In [1]:
import pandas as pd
import json
import time
from pathlib import Path

import os
import sys

sys.path.append(os.path.abspath("preprocessing"))

import preprocessing.preprocess_pipeline as pp
from preprocessing.pipeline_io import save_pipeline, load_pipeline

from optuna import load_study

from train_catboost_optuna import train_and_save_best, detect_cat_cols

In [ ]:
artifacts = Path('artifacts')
artifacts.mkdir(parents=True, exist_ok=True)

TRAIN_DATA_PATH = 'data/train_data.parquet'

PIPELINE_PATH = artifacts / 'prep_v1'
USE_LOCAL_PIPELINE = True

COLS_TO_DROP_EARLY = ['subtitle','differential_pricing','international_delivery_mode','listing_source','site_id', 'coverage_areas']
DATE_COLS = ['date_created','last_updated']
TARGET = "condition"

USE_LOCAL_BEST_PARAMS = True

STUDY_NAME = 'meli_used_new'
PRIVATE_STORAGE_URL = ''

In [ ]:
df_train = pd.read_parquet(
    TRAIN_DATA_PATH,
)

df_train = df_train.drop(columns=COLS_TO_DROP_EARLY).copy()

In [4]:
if not USE_LOCAL_PIPELINE:
    pipe = pp.FeaturePipeline().fit(df_train)
    save_pipeline(pipe, out_dir=PIPELINE_PATH)

In [5]:
pipe_loaded, _ = load_pipeline(PIPELINE_PATH)
y = df_train[TARGET].eq('new').astype(int)                
X = pipe_loaded.transform(df_train)

In [6]:
cat_cols = detect_cat_cols(X, exclude=[])
other_cat_cols = ['listing_type_id', 'category_level1', 'price_bin10', 'category_id', 'seller_id']
cat_cols = list(dict.fromkeys(cat_cols + other_cat_cols))

X = X.loc[:, X.notna().any(axis=0)].copy()
for c in cat_cols:
    if c in X.columns:
        X[c] = X[c].astype("object")

In [7]:
if USE_LOCAL_BEST_PARAMS:
    with open("artifacts/best_params.json", "r") as f:
        best_params = json.load(f)
else:
    study = load_study(
        study_name=STUDY_NAME,
        storage=PRIVATE_STORAGE_URL
    )
    best_params = study.best_params

    (artifacts / "best_params.json").write_text(json.dumps(best_params, indent=2))
    (artifacts / "study_best.json").write_text(json.dumps({
        "value": study.best_value,
        "number": study.best_trial.number,
        "user_attrs": study.best_trial.user_attrs,
    }, indent=2))

In [8]:
# Normalize best params for CatBoost (explicit Nones)
if "weighting_mode" in best_params:
    wm = best_params["weighting_mode"]
    if wm == "none":
        best_params["auto_class_weights"] = None
        best_params["scale_pos_weight"] = None
    elif wm == "auto_balanced":
        best_params["auto_class_weights"] = "Balanced"
        best_params["scale_pos_weight"] = None
    elif wm == "scale_pos_weight":
        best_params["auto_class_weights"] = None
    del best_params["weighting_mode"]

In [ ]:
best_params

In [ ]:
summary = train_and_save_best(
    X=X,
    y=y,
    groups=None,
    cat_cols=cat_cols,
    best_params=best_params,
    n_splits=2,
    seed=int(time.time()) % 10_000_000,
    early_stopping_rounds=200,
    thread_count=-1,
    artifacts_dir=artifacts,
    log_every_iter=100,
)

In [ ]:
print("\n=== Done (Training) ===")
print("CV mean accuracy:", summary["cv_metrics_mean"].get("accuracy"))
print(f"Threshold metric: {summary['threshold_metric']}")
print("Best threshold:", summary["best_threshold"], "Value:", summary["best_threshold_value"])
if summary["oof_metrics_calibrated"] is not None:
    print("OOF metrics (calibrated):", summary["oof_metrics_calibrated"])
    print("Best threshold (calibrated):", summary["best_threshold_calibrated"], "Value:", summary["best_threshold_calibrated_value"])
if summary.get("segment_threshold_cols"):
    print("Segment thresholds learned for:", summary["segment_threshold_cols"])